# Social Media Usage and Emotional Well-Being
## Classification using ML and DL

This notebook contains:
- Random Forest Classifier (ML)
- Artificial Neural Network (ANN) (DL)
- Data preprocessing
- Model evaluation

In [ ]:
# Install required libraries (Run if needed)
# !pip install pandas numpy scikit-learn tensorflow

In [ ]:
# =========================
# IMPORT LIBRARIES
# =========================

import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

In [ ]:
# =========================
# LOAD DATASETS
# =========================

train_df = pd.read_csv(
    "train.csv",
    engine='python',
    on_bad_lines='skip'
)

val_df = pd.read_csv(
    "validation.csv",
    engine='python',
    on_bad_lines='skip'
)

test_df = pd.read_csv(
    "test.csv",
    engine='python',
    on_bad_lines='skip'
)

print(train_df.head())

In [ ]:
# =========================
# REMOVE MISSING VALUES
# =========================

train_df = train_df.dropna()
val_df = val_df.dropna()
test_df = test_df.dropna()

print("Train Shape:", train_df.shape)
print("Validation Shape:", val_df.shape)
print("Test Shape:", test_df.shape)

In [ ]:
# =========================
# TARGET COLUMN
# =========================

target = "Dominant_Emotion"

# Features and labels
X_train = train_df.drop(target, axis=1)
y_train = train_df[target]

X_val = val_df.drop(target, axis=1)
y_val = val_df[target]

X_test = test_df.drop(target, axis=1)
y_test = test_df[target]

In [ ]:
# =========================
# ENCODE CATEGORICAL FEATURES
# =========================

for col in X_train.columns:

    if X_train[col].dtype == 'object':

        le = LabelEncoder()

        combined = pd.concat([
            X_train[col],
            X_val[col],
            X_test[col]
        ]).astype(str)

        le.fit(combined)

        X_train[col] = le.transform(X_train[col].astype(str))
        X_val[col] = le.transform(X_val[col].astype(str))
        X_test[col] = le.transform(X_test[col].astype(str))

# Encode target labels
target_encoder = LabelEncoder()

combined_target = pd.concat([
    y_train,
    y_val,
    y_test
]).astype(str)

target_encoder.fit(combined_target)

y_train = target_encoder.transform(y_train.astype(str))
y_val = target_encoder.transform(y_val.astype(str))
y_test = target_encoder.transform(y_test.astype(str))

print("Encoded successfully")

In [ ]:
# =========================
# FEATURE SCALING
# =========================

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

print("Feature scaling completed")

## Random Forest Classifier (ML Technique)

In [ ]:
# =========================
# RANDOM FOREST MODEL
# =========================

rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

# Train model
rf_model.fit(X_train, y_train)

# Validation predictions
val_pred = rf_model.predict(X_val)

# Test predictions
test_pred = rf_model.predict(X_test)

# Accuracy
val_accuracy = accuracy_score(y_val, val_pred)
test_accuracy = accuracy_score(y_test, test_pred)

print("Validation Accuracy:", val_accuracy)
print("Test Accuracy:", test_accuracy)

# Classification report
print("\nClassification Report:\n")
print(classification_report(y_test, test_pred))

## Artificial Neural Network (DL Technique)

In [ ]:
# =========================
# BUILD ANN MODEL
# =========================

ann_model = Sequential()

ann_model.add(Dense(
    64,
    activation='relu',
    input_shape=(X_train.shape[1],)
))

ann_model.add(Dense(32, activation='relu'))

ann_model.add(Dense(
    len(np.unique(y_train)),
    activation='softmax'
))

# Compile model
ann_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

ann_model.summary()

In [ ]:
# =========================
# TRAIN ANN MODEL
# =========================

history = ann_model.fit(
    X_train,
    y_train,
    epochs=30,
    batch_size=16,
    validation_data=(X_val, y_val)
)

In [ ]:
# =========================
# EVALUATE ANN MODEL
# =========================

loss, accuracy = ann_model.evaluate(X_test, y_test)

print("\nTest Accuracy:", accuracy)

# Predictions
predictions = ann_model.predict(X_test)

predicted_classes = np.argmax(predictions, axis=1)

print("\nPredicted Classes:")
print(predicted_classes[:10])